In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


In [9]:
# =========================================
# 1) 데이터 로드
# =========================================
df = pd.read_csv("/content/drive/MyDrive/mhealth_dataset_combined.csv")   # 파일명에 맞게 변경
FEATS = [c for c in df.columns if c not in ["label", "person"]]

NUM_CLASSES = df["label"].nunique()
seq_len = 200
df

,chest_acc_x,chest_acc_y,chest_acc_z,ecg_1,ecg_2,l_ankle_acc_x,l_ankle_acc_y,l_ankle_acc_z,l_ankle_gyro_x,l_ankle_gyro_y,...,r_arm_acc_y,r_arm_acc_z,r_arm_gyro_x,r_arm_gyro_y,r_arm_gyro_z,r_arm_mag_x,r_arm_mag_y,r_arm_mag_z,label,subject
0,-9.8184,0.009971,0.295630,0.004186,0.004186,2.18490,-9.6967,0.63077,0.103900,-0.84053,...,-4.5781,0.187760,-0.44902,-1.01030,0.034483,-2.35000,-1.610200,-0.030899,0,1
1,-9.8489,0.524040,0.373480,0.004186,0.016745,2.38760,-9.5080,0.68389,0.085343,-0.83865,...,-4.3198,0.023595,-0.44902,-1.01030,0.034483,-2.16320,-0.882540,0.326570,0,1
2,-9.6602,0.181850,0.437420,0.016745,0.037677,2.40860,-9.5674,0.68113,0.085343,-0.83865,...,-4.2772,0.275720,-0.44902,-1.01030,0.034483,-1.61750,-0.165620,-0.030693,0,1
3,-9.6507,0.214220,0.240330,0.079540,0.117220,2.18140,-9.4301,0.55031,0.085343,-0.83865,...,-4.3163,0.367520,-0.45686,-1.00820,0.025862,-1.07710,0.006945,-0.382620,0,1
4,-9.7030,0.303890,0.311560,0.221870,0.205130,2.41730,-9.3889,0.71098,0.085343,-0.83865,...,-4.1459,0.407290,-0.45686,-1.00820,0.025862,-0.53684,0.175900,-1.095500,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1215740,-12.2440,-2.970600,-0.035772,0.812140,1.096800,0.57061,-2.5602,0.41936,-0.055659,0.64165,...,-16.9660,2.370400,0.10980,-0.99384,0.191810,-6.12870,15.495000,18.856000,0,9
1215741,-10.9220,-3.419000,-0.339280,1.469400,1.285200,5.26840,-4.9331,9.66020,-0.055659,0.64165,...,-13.3900,2.997200,0.10980,-0.99384,0.191810,-10.24200,17.139000,33.920000,0,9
1215742,-9.4842,-3.064300,-1.033700,0.238620,0.891680,0.53805,-5.9706,2.93600,-0.055659,0.64165,...,-11.3790,3.149800,0.10980,-0.99384,0.191810,-12.94400,16.170000,43.262000,0,9
1215743,-8.7889,-2.475700,-0.612290,-0.205130,0.460490,1.56950,-7.9809,-2.05000,-0.085343,0.43715,...,-10.0000,3.000000,0.11961,-0.97331,0.153020,-14.52400,1.849400,43.373000,0,9


In [10]:
df["person"] = df["subject"]


In [11]:
# =========================================
# 2) Robust Scaling 직접 구현
# =========================================
def robust_fit(train_df):
    arr = train_df[FEATS].to_numpy()
    med = np.median(arr, axis=0)
    q1 = np.percentile(arr, 25, axis=0)
    q3 = np.percentile(arr, 75, axis=0)
    iqr = q3 - q1
    iqr[iqr == 0] = 1.0
    return med, iqr

def robust_transform(df_part, med, iqr):
    return (df_part[FEATS].to_numpy() - med) / iqr


In [12]:
# =========================================
# 3) 시퀀스 자르기 (증강 아님)
# =========================================
def make_sequences(df_part):
    X_list, y_list = [], []
    arr = df_part[FEATS].to_numpy()
    labels = df_part["label"].to_numpy()

    for i in range(0, len(df_part) - seq_len, seq_len):
        X_list.append(arr[i:i+seq_len])
        y_list.append(labels[i+seq_len//2])

    return np.array(X_list), np.array(y_list)

In [13]:
# =========================================
# 4) DropPath (Stochastic Depth)
# =========================================
class DropPath(layers.Layer):
    def __init__(self, drop_prob=0.1):
        super().__init__()
        self.drop_prob = drop_prob

    def call(self, x, training=False):
        if (not training) or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        mask = tf.cast(tf.random.uniform(tf.shape(x)) < keep_prob, x.dtype)
        return x * mask / keep_prob


In [14]:
# =========================================
# 5) Depthwise Conv Block
# =========================================
def depth_block(x, filters):
    x = layers.DepthwiseConv1D(5, padding='same')(x)
    x = layers.Conv1D(filters, 1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

In [15]:
# =========================================
# 6) 최종 모델
# =========================================
def build_model(seq_len, n_features, n_classes):

    inp = layers.Input(shape=(seq_len, n_features))

    x = depth_block(inp, 64)
    x = DropPath(0.1)(x)

    x = depth_block(x, 128)
    x = DropPath(0.1)(x)

    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(n_classes, activation='softmax')(x)

    model = Model(inp, out)

    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
    opt = tf.keras.optimizers.Adam(1e-3)

    model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])
    return model


In [16]:
# =========================================
# 7) Cross-person folds
# =========================================
folds = [(1,2), (3,4), (5,6), (7,8), (9,10)]
all_preds, all_trues = [], []

for (p1, p2) in folds:
    print(f"\n==== Fold | Test=[{p1}, {p2}] ====")

    df_train = df[df["person"].isin([p for p in range(1,11) if p not in [p1,p2]])]
    df_test  = df[df["person"].isin([p1,p2])]

    # Robust Scaler fit
    med, iqr = robust_fit(df_train)

    # scaling
    df_train_scaled = df_train.copy()
    df_test_scaled  = df_test.copy()
    df_train_scaled[FEATS] = robust_transform(df_train, med, iqr)
    df_test_scaled[FEATS]  = robust_transform(df_test, med, iqr)

    # make sequences
    X_train, y_train = make_sequences(df_train_scaled)
    X_test,  y_test  = make_sequences(df_test_scaled)

    print("Train:", X_train.shape, "| Test:", X_test.shape)

    # one-hot
    y_train_oh = tf.one_hot(y_train, NUM_CLASSES)
    y_test_oh  = tf.one_hot(y_test, NUM_CLASSES)

    # class weights (불균형 완화)
    cw = {
        1:5.0, 2:5.0, 3:4.0, 4:3.5,
        10:1.5, 12:2.0
    }

    model = build_model(seq_len, len(FEATS), NUM_CLASSES)

    callback = tf.keras.callbacks.EarlyStopping(
        patience=4,
        monitor="val_accuracy",
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train_oh,
        validation_data=(X_test, y_test_oh),
        epochs=18,
        batch_size=64,
        class_weight=cw,
        callbacks=[callback],
        verbose=1
    )

    preds = np.argmax(model.predict(X_test), axis=1)

    all_preds.extend(preds)
    all_trues.extend(y_test)


==== Fold | Test=[1, 2] ====
Train: (4619, 200, 24) | Test: (1459, 200, 24)
Epoch 1/18
73/73 ━━━━━━━━━━━━━━━━━━━━ 25s 150ms/step - accuracy: 0.6383 - loss: 2.3012 - val_accuracy: 0.7724 - val_loss: 2.1488
Epoch 2/18
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7129 - loss: 1.3356 - val_accuracy: 0.7327 - val_loss: 1.8209
Epoch 3/18
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8045 - loss: 1.0096 - val_accuracy: 0.7245 - val_loss: 1.5398
Epoch 4/18
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8150 - loss: 0.9356 - val_accuracy: 0.7670 - val_loss: 1.2024
Epoch 5/18
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8482 - loss: 0.8624 - val_accuracy: 0.7512 - val_loss: 0.9999
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step

==== Fold | Test=[3, 4] ====
Train: (4884, 200, 24) | Test: (1194, 200, 24)
Epoch 1/18
77/77 ━━━━━━━━━━━━━━━━━━━━ 19s 142ms/step - accuracy: 0.6925 - loss: 2.2182 - val_accuracy: 0.7060 - val_loss: 2.2330
Epoch 2/18
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s

In [17]:
# =========================================
# 8) 최종 결과 출력
# =========================================
print("\n===== FINAL SCORE =====")
print("Accuracy:", accuracy_score(all_trues, all_preds))
print("Weighted F1:", f1_score(all_trues, all_preds, average="weighted"))

print("\nConfusion Matrix:")
print(confusion_matrix(all_trues, all_preds))

print("\nClassification Report:")
print(classification_report(all_trues, all_preds))


===== FINAL SCORE =====
Accuracy: 0.7896988645713345
Weighted F1: 0.7735819328111175

Confusion Matrix:
[[3951   69   27   58   33   16   36   18   24   50   24   30   26]
 [ 113   42    0    0    0    0    0    0    0    0    0    0    0]
 [ 109   15   15    0    0    0    0    0   16    0    0    0    0]
 [  15    0    0  138    0    0    0    0    0    0    0    0    0]
 [  94    0    0    0   60    0    0    0    0    0    0    0    0]
 [  88    0    0    0   10   56    0    0    0    0    0    0    0]
 [  48    0    0    0    0    0   91    0    0    0    0    0    0]
 [  85    0    0    0    0    0    0   61    0    0    0    0    0]
 [  90    0    0    0    0    1    0    0   57    0    0    0    0]
 [  66    0    0    0    0    0    0    0    0   89    0    0    0]
 [  43    0    0    0    0    0    0    0    0    0   97   14    0]
 [  39    0    0    0    0    0    0    0    0    0    1  110    0]
 [  20    0    0    0    0    0    0    0    0    0    0    0   32]]

Classific